<a href="https://colab.research.google.com/github/armandoordonez/AI-Engineering/blob/main/Knowledge_Distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Knowledge Distillation Benchmark
This notebook demonstrates the practical differences between a large **Teacher** model and smaller **Student** models. We will measure size, speed (latency), and the similarity of their predictions (Soft Labels).


In [ ]:
# Install necessary libraries
!pip install transformers torch pandas

## 1. Environment Setup and Model Loading
We initialize the models from Hugging Face. We use **BERT-base** as our Teacher and distilled version: **DistilBERT** (Standard)


In [ ]:
import torch
import time
import os
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Model selection from Hugging Face
models_to_bench = {
    "Teacher (BERT)": "textattack/bert-base-uncased-SST-2",
    "Challenge Winner (TinyBERT)": "huawei-noah/TinyBERT_General_4L_312D",
    # "WINNER MODEL": "<ENTER MODEL NAME HERE>"
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Pre-loading tokenizers and models for the benchmark
loaded_models = {}
for label, model_id in models_to_bench.items():
    print(f"Loading {label}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    # Using 2 labels for a simple binary classification structure
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2).to(device)
    loaded_models[label] = (model, tokenizer)


Using device: cpu
Loading Teacher (BERT)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading Challenge Winner (TinyBERT)...


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not o

## 2. Benchmark Utility Functions
We define functions to calculate the number of parameters, the physical size of the model on disk, and the inference latency.


In [ ]:
def count_parameters(model):
    """Calculates total parameters in Millions."""
    return sum(p.numel() for p in model.parameters()) / 1e6

def get_model_size(model):
    """Saves a temporary file to measure size in MB."""
    temp_path = "temp_model.p"
    torch.save(model.state_dict(), temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    os.remove(temp_path)
    return size_mb

def measure_latency(model, tokenizer, text, runs=100):
    """Measures average inference time in milliseconds."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    model.eval()

    # Warmup runs to initialize GPU/CPU state
    with torch.no_grad():
        for _ in range(10):
            _ = model(**inputs)

    start_time = time.time()
    with torch.no_grad():
        for _ in range(runs):
            _ = model(**inputs)
    end_time = time.time()

    return ((end_time - start_time) / runs) * 1000


## 3. Execution and Competitive Scoring
In this section, we run the test sentence through all models and compute the **Efficiency Score**.
**Score Formula:** (1 / Latency) * (1 / Size) * 1000


In [ ]:
test_sentence = "No es cierto que el modelo no sea lento"
results = []
logits_comparison = {}

for label, (model, tokenizer) in loaded_models.items():
    params = count_parameters(model)
    size_mb = get_model_size(model)
    latency = measure_latency(model, tokenizer, test_sentence)

    # Efficiency calculation
    efficiency_score = (1 / latency) * (1 / size_mb) * 1000

    results.append({
        "Model": label,
        "Params (M)": round(params, 2),
        "Size (MB)": round(size_mb, 2),
        "Latency (ms)": round(latency, 2),
        "Efficiency Score": round(efficiency_score, 4)
    })

    # Capture Soft Labels (Logits) for comparison
    inputs = tokenizer(test_sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits_comparison[label] = outputs.logits.cpu().numpy()[0]

# Generate comparison table
df = pd.DataFrame(results)
print(df.to_string(index=False))


                      Model  Params (M)  Size (MB)  Latency (ms)  Efficiency Score
             Teacher (BERT)      109.48     417.72        115.19            0.0208
Challenge Winner (TinyBERT)       14.35      54.77          9.10            2.0074


## 4. Knowledge Preservation Analysis
The core of Knowledge Distillation is how well the Student mimics the Teacher's probabilities (**Soft Labels**). Here we compare the raw output values (logits).


In [ ]:
print("Soft Labels (Logits) Comparison:")
logits_df = pd.DataFrame(logits_comparison, index=["Class 0", "Class 1"])
print(logits_df)

Soft Labels (Logits) Comparison:
         Teacher (BERT)  Challenge Winner (TinyBERT)
Class 0        0.208130                    -0.000341
Class 1       -0.098658                     0.002739


## 5. Visualizing the Effect of Temperature ($T$)
Temperature is a hyperparameter that softens the probability distribution. High temperatures reveal the "dark knowledge" of the Teacher, showing which classes are similar to each other. Low temperatures make the model more confident but rigid.


In [ ]:
import torch.nn.functional as F

def visualize_temperature(logits, temperatures=[1, 2, 5, 10]):
    print("Effect of Temperature on Teacher's Soft Labels:")
    # We use the logits captured in the previous step
    teacher_logits = torch.tensor(logits)

    temp_data = {}
    for T in temperatures:
        soft_probs = F.softmax(teacher_logits / T, dim=0)
        temp_data[f"T={T}"] = soft_probs.numpy().round(4)

    temp_df = pd.DataFrame(temp_data, index=["Class 0", "Class 1"])
    print(temp_df)

# Analyze the Teacher's knowledge distribution
visualize_temperature(logits_comparison["Teacher (BERT)"])


Effect of Temperature on Teacher's Soft Labels:
            T=1     T=2     T=5    T=10
Class 0  0.5761  0.5383  0.5153  0.5077
Class 1  0.4239  0.4617  0.4847  0.4923


### 6. Mini-Challenge: Shinking BERT into a Nano-Student

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertForSequenceClassification, BertConfig, AutoTokenizer

test_phrase = "I cannot believe how good is this movie"

# Load Teacher (BERT) and initialize the empty Nano-Student
teacher = loaded_models["Teacher (BERT)"][0]
teacher.eval()
student = BertForSequenceClassification(nano_config).to(device)

# --- NEW SECTION: THE PRE-DISTILLATION BASELINE ---
test_phrase = "I hate how much I liked this show."
inputs_baseline = tokenizer(test_phrase, return_tensors="pt").to(device)

student.eval() # Modo evaluación para el baseline
with torch.no_grad():
    baseline_logits = student(**inputs_baseline).logits
    baseline_probs = F.softmax(baseline_logits, dim=-1)

print("PRE-DISTILLATION CHECK (The Baseline)")
print(f"Phrase: '{test_phrase}'")
print(f"Nano-Student Initial Confidence (Pos): {baseline_probs[0][1]:.4f}")
print("Status: The model is guessing randomly (untrained).")
print("-" * 50)

# --- 2. THE DISTILLATION ENGINE ---
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)
T = 8.0      # Temperature to reveal 'Dark Knowledge'
alpha = 0.7  # Balance: 70% mimicking teacher, 30% learning from ground truth

# Mini-dataset for the live demo
texts = [
    "I love this movie, it's absolutely fantastic!",
    "This movie will be terrible"
]
labels = torch.tensor([1, 0]).to(device)


inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)

print(f"Distilling BERT into Nano-Student (Params: {student.num_parameters()/1e6:.2f}M)...")
print(f"{'Epoch':<10} | {'Total Loss':<15} | {'KL Knowledge Gap':<15}")
print("-" * 50)

# --- 3. LIVE TRAINING LOOP ---
for epoch in range(51):
    student.train()
    optimizer.zero_grad()

    # Get Teacher's 'Wisdom' (Logits)
    with torch.no_grad():
        teacher_logits = teacher(**inputs).logits

    # Get Student's 'Attempt'
    student_logits = student(**inputs).logits

    # KL Divergence: The core of Knowledge Distillation
    # We compare how the Student's 'soft' distribution matches the Teacher's
    soft_targets = F.softmax(teacher_logits / T, dim=-1)
    soft_log_probs = F.log_softmax(student_logits / T, dim=-1)

    # Loss 1: Imitation (KL) | Loss 2: Accuracy (CrossEntropy)
    distillation_loss = nn.KLDivLoss(reduction='batchmean')(soft_log_probs, soft_targets) * (T**2)
    student_loss = F.cross_entropy(student_logits, labels)

    total_loss = (alpha * distillation_loss) + (1.0 - alpha) * student_loss

    total_loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"{epoch:<10} | {total_loss.item():<15.4f} | {distillation_loss.item():<15.4f}")

print("-" * 50)
print("Distillation Complete! The student has captured the Teacher's probability distribution.")

PRE-DISTILLATION CHECK (The Baseline)
Phrase: 'I hate how much I liked this show.'
Nano-Student Initial Confidence (Pos): 0.4834
Status: The model is guessing randomly (untrained).
--------------------------------------------------
Distilling BERT into Nano-Student (Params: 5.70M)...
Epoch      | Total Loss      | KL Knowledge Gap
--------------------------------------------------
0          | 4.6129          | 6.2984         
10         | 4.6311          | 6.3187         
20         | 4.5349          | 6.1952         
30         | 4.5519          | 6.2155         
40         | 4.4867          | 6.1327         
50         | 4.1466          | 5.6965         
--------------------------------------------------
Distillation Complete! The student has captured the Teacher's probability distribution.


### 6.1 Verifying

In [ ]:
inputs_test = tokenizer(test_phrase, return_tensors="pt").to(device)
student.eval()
teacher.eval()

with torch.no_grad():
    t_logits = teacher(**inputs_test).logits
    s_logits = student(**inputs_test).logits

    t_probs = F.softmax(t_logits, dim=-1)
    s_probs = F.softmax(s_logits, dim=-1)

print(f"Test Phrase: '{test_phrase}'")
print("-" * 30)
print(f"Teacher Confidence (Pos): {t_probs[0][1]:.4f}")
print(f"Student Confidence (Pos): {s_probs[0][1]:.4f}")
print("-" * 30)

if torch.argmax(t_probs) == torch.argmax(s_probs):
    print("SUCCESS: The Student mimicked the Teacher's decision!")
else:
    print("FAILURE: The Student is could not capture this logic.")

Test Phrase: 'I hate how much I liked this show.'
------------------------------
Teacher Confidence (Pos): 0.1947
Student Confidence (Pos): 0.5563
------------------------------
FAILURE: The Student is could not capture this logic.
